# Fine-Tuning a Vision Transformer (ViT) for Blood Cell Classification

This notebook fine-tunes `google/vit-base-patch16-224` on the
[`ehottl/blood_dataset`](https://huggingface.co/datasets/ehottl/blood_dataset)
(46,232 microscope images across 8 blood cell classes: basophil, eosinophil,
erythroblast, lymphocyte, monocyte, neutrophil, platelet, plasma cell).

**Pipeline:** load & split data → preprocess with `ViTImageProcessor` →
fine-tune with `Trainer` → evaluate (accuracy, F1, confusion matrix) →
push to Hub (optional) → interactive Gradio demo (optional).

Runs end-to-end on a free Colab T4 GPU. Runtime: **GPU** (Runtime → Change runtime type → T4 GPU).


## 1. Setup

If you opened this notebook standalone (e.g. via "Open in Colab" badge without
cloning the repo), run the clone cell below so relative paths to
`../configs/config.yaml` and `../results/` resolve correctly. Skip it if
you've already mounted/cloned the repo another way.

In [ ]:
# Optional: clone the repo so configs/ and results/ are available at ../
import os

REPO_URL = "https://githubtocolab.com/Praruj/vit-biomedical-finetuning.git"

if not os.path.exists("/content/vit-biomedical-finetuning") and "google.colab" in str(get_ipython()):
    !git clone -q {REPO_URL}
    %cd vit-biomedical-finetuning/notebooks

### Mount Google Drive (recommended)

Colab wipes local disk when the runtime disconnects — mounting Drive means
your trained model checkpoint survives even if the session times out or you
close the tab. Skip this cell if you don't mind retraining from scratch each
session.

In [ ]:
USE_DRIVE = True  # set False to keep everything local to the Colab runtime (not persisted)

if USE_DRIVE and "google.colab" in str(get_ipython()):
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/vit-biomedical-finetuning/results"
    print(f"Model checkpoints will be saved to: {DRIVE_OUTPUT_DIR}")
else:
    DRIVE_OUTPUT_DIR = None
    print("Not using Drive — output will only exist for this Colab session.")

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn gradio torch torchvision pyyaml

### Mount Google Drive (persist your trained model!)

Colab wipes local disk when the runtime disconnects — if you don't save
to Drive, a trained model is gone the moment the session ends. Run this
cell once per session; it'll ask you to authorize access.

In [ ]:
IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/vit-biomedical-finetuning/results"
    print(f"Drive mounted. Model checkpoints will be saved to: {DRIVE_OUTPUT_DIR}")
else:
    DRIVE_OUTPUT_DIR = None  # running locally — config.yaml's output_dir is used as-is

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

### Load config

All hyperparameters live in `configs/config.yaml` rather than hardcoded in the
notebook, so the run is easy to reproduce/tweak without touching code. If
you're running this in Colab without cloning the repo, the fallback default
config below is used instead.

In [ ]:
import yaml
import os

CONFIG_PATH = "../configs/config.yaml"  # relative path if repo is cloned into Colab

default_config = {
    "model": {"checkpoint": "google/vit-base-patch16-224", "num_labels": 8},
    "dataset": {"name": "ehottl/blood_dataset", "train_split": 0.8, "val_split": 0.1,
                "test_split": 0.1, "seed": 42},
    "preprocessing": {"image_size": 224},
    "training": {
        "output_dir": "results/vit-blood-cell-classifier",
        "per_device_train_batch_size": 32,
        "per_device_eval_batch_size": 32,
        "gradient_accumulation_steps": 1,
        "num_train_epochs": 4,
        "learning_rate": 2e-5,
        "warmup_ratio": 0.1,
        "weight_decay": 0.01,
        "fp16": True,
        "eval_strategy": "epoch",
        "save_strategy": "epoch",
        "logging_steps": 50,
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1_macro",
        "save_total_limit": 2,
    },
    "hub": {"push_to_hub": False, "repo_id": "your-hf-username/vit-blood-cell-classifier"},
}

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        cfg = yaml.safe_load(f)
    print(f"Loaded config from {CONFIG_PATH}")
else:
    cfg = default_config
    print("configs/config.yaml not found — using built-in default config "
          "(clone the full repo into Colab to use configs/config.yaml directly).")

# Save checkpoints to Drive if mounted, so training survives a disconnect
if DRIVE_OUTPUT_DIR:
    cfg["training"]["output_dir"] = DRIVE_OUTPUT_DIR

cfg

### (Optional) Hugging Face login

Only needed if you plan to **push the fine-tuned model to the Hub**.
Skip this cell entirely if you just want to train and evaluate locally in Colab.

In [ ]:
PUSH_TO_HUB = cfg["hub"]["push_to_hub"]  # flip to True in configs/config.yaml to upload

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login
    notebook_login()

## 2. Load the dataset

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(cfg["dataset"]["name"])
print(raw_dataset)

The dataset only ships a single `train` split (46,232 examples), so we
create our own train / validation / test splits (80 / 10 / 10).

In [ ]:
ds_cfg = cfg["dataset"]
val_test_size = ds_cfg["val_split"] + ds_cfg["test_split"]
test_of_holdout = ds_cfg["test_split"] / val_test_size  # relative fraction within the holdout

split_1 = raw_dataset["train"].train_test_split(test_size=val_test_size, seed=ds_cfg["seed"], stratify_by_column="label")
split_2 = split_1["test"].train_test_split(test_size=test_of_holdout, seed=ds_cfg["seed"], stratify_by_column="label")

from datasets import DatasetDict

dataset = DatasetDict({
    "train": split_1["train"],
    "validation": split_2["train"],
    "test": split_2["test"],
})
dataset

In [ ]:
labels = dataset["train"].features["label"].names
num_labels = len(labels)
id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in enumerate(labels)}

print(f"{num_labels} classes:", labels)

In [ ]:
# Quick peek at class balance in the training split
import pandas as pd

counts = pd.Series(dataset["train"]["label"]).value_counts().rename(index=id2label)
counts

In [ ]:
# Visualize a few samples
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax in axes.flatten():
    idx = torch.randint(0, len(dataset["train"]), (1,)).item()
    example = dataset["train"][idx]
    ax.imshow(example["image"])
    ax.set_title(id2label[example["label"]])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
from transformers import ViTImageProcessor

model_checkpoint = cfg["model"]["checkpoint"]
processor = ViTImageProcessor.from_pretrained(model_checkpoint)
processor

In [ ]:
from torchvision.transforms import (
    Compose, Normalize, RandomHorizontalFlip, RandomRotation,
    RandomResizedCrop, Resize, CenterCrop, ToTensor,
)

image_mean, image_std = processor.image_mean, processor.image_std
size = cfg["preprocessing"].get("image_size", processor.size["height"])

normalize = Normalize(mean=image_mean, std=image_std)

train_transforms = Compose([
    RandomResizedCrop(size),
    RandomHorizontalFlip(),
    RandomRotation(10),
    ToTensor(),
    normalize,
])

eval_transforms = Compose([
    Resize(size),
    CenterCrop(size),
    ToTensor(),
    normalize,
])

def apply_train_transforms(batch):
    batch["pixel_values"] = [train_transforms(img.convert("RGB")) for img in batch["image"]]
    return batch

def apply_eval_transforms(batch):
    batch["pixel_values"] = [eval_transforms(img.convert("RGB")) for img in batch["image"]]
    return batch

dataset["train"].set_transform(apply_train_transforms)
dataset["validation"].set_transform(apply_eval_transforms)
dataset["test"].set_transform(apply_eval_transforms)

In [ ]:
def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["label"] for item in batch])
    return {"pixel_values": pixel_values, "labels": labels}

## 4. Load the model

In [ ]:
from transformers import ViTForImageClassification

assert num_labels == cfg["model"]["num_labels"], (
    f"Dataset has {num_labels} classes but config.yaml specifies "
    f"{cfg['model']['num_labels']} — update configs/config.yaml."
)

model = ViTForImageClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # replaces the original 1000-class head
)

## 5. Metrics

In [ ]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    references = eval_pred.label_ids
    acc = accuracy_metric.compute(predictions=predictions, references=references)
    f1 = f1_metric.compute(predictions=predictions, references=references, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

## 6. Training

Settings below are tuned for a free-tier Colab **T4 (16GB)** GPU:
- `fp16=True` for mixed precision (much faster, lower memory on T4)
- Small-ish batch size with gradient accumulation to keep memory in check
- `remove_unused_columns=False` because we handle the image/label mapping ourselves via `set_transform`


In [ ]:
from transformers import TrainingArguments, Trainer

t_cfg = cfg["training"]
output_dir = t_cfg["output_dir"]

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=t_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=t_cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=t_cfg["gradient_accumulation_steps"],
    num_train_epochs=t_cfg["num_train_epochs"],
    learning_rate=float(t_cfg["learning_rate"]),
    warmup_ratio=t_cfg["warmup_ratio"],
    weight_decay=t_cfg["weight_decay"],
    fp16=t_cfg["fp16"] and torch.cuda.is_available(),
    eval_strategy=t_cfg["eval_strategy"],
    save_strategy=t_cfg["save_strategy"],
    logging_steps=t_cfg["logging_steps"],
    load_best_model_at_end=t_cfg["load_best_model_at_end"],
    metric_for_best_model=t_cfg["metric_for_best_model"],
    save_total_limit=t_cfg["save_total_limit"],
    remove_unused_columns=False,
    report_to="none",
    push_to_hub=False,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    tokenizer=processor,
)

In [ ]:
train_results = trainer.train()
trainer.save_model()
trainer.log_metrics("train", train_results.metrics)
trainer.save_metrics("train", train_results.metrics)

## 7. Evaluation on the held-out test set

In [ ]:
test_results = trainer.predict(dataset["test"])
print(test_results.metrics)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_true = test_results.label_ids
y_pred = np.argmax(test_results.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=labels))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix — ViT Blood Cell Classifier (test set)")
plt.tight_layout()

results_dir = cfg["training"]["output_dir"]  # same location as saved model checkpoint
os.makedirs(results_dir, exist_ok=True)
plt.savefig(os.path.join(results_dir, "confusion_matrix.png"), dpi=150)
plt.show()

### A few misclassified examples

Useful for the portfolio write-up / README: show a handful of the model's mistakes.

In [ ]:
misclassified_idx = np.where(y_true != y_pred)[0][:8]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flatten(), misclassified_idx):
    example = dataset["test"][int(idx)]
    ax.imshow(example["image"])
    ax.set_title(f"true: {id2label[y_true[idx]]}\npred: {id2label[y_pred[idx]]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Save / push the model

The model was already saved locally to `./vit-blood-cell-classifier` by `trainer.save_model()`.
Pushing to the Hub is optional and only runs if you logged in and set `PUSH_TO_HUB = True` above.

In [ ]:
if PUSH_TO_HUB:
    repo_name = cfg["hub"]["repo_id"]  # set in configs/config.yaml
    trainer.push_to_hub(repo_name)
    processor.push_to_hub(repo_name)
    print(f"Model pushed to: https://huggingface.co/{repo_name}")
else:
    print("Skipping push_to_hub (PUSH_TO_HUB=False). Model is saved locally at:", output_dir)

## 9. (Optional) Gradio demo

A minimal interactive demo: upload/drag a blood cell image, get the predicted class
and confidence scores. Great for embedding a live demo link in your portfolio
(Gradio can also be deployed for free on Hugging Face Spaces).

In [ ]:
import gradio as gr
from PIL import Image
import torch.nn.functional as F

inference_model = trainer.model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
inference_model.to(device)

def classify_blood_cell(image: Image.Image):
    image = image.convert("RGB")
    pixel_values = eval_transforms(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = inference_model(pixel_values=pixel_values).logits
        probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
    return {id2label[i]: float(probs[i]) for i in range(num_labels)}

demo = gr.Interface(
    fn=classify_blood_cell,
    inputs=gr.Image(type="pil", label="Upload a blood cell image"),
    outputs=gr.Label(num_top_classes=8, label="Predicted class"),
    title="ViT Blood Cell Classifier",
    description=(
        "Fine-tuned google/vit-base-patch16-224 on the ehottl/blood_dataset "
        "(8 classes: basophil, eosinophil, erythroblast, lymphocyte, monocyte, "
        "neutrophil, plasma cell, platelet)."
    ),
    examples=None,
)

demo.launch(share=True, debug=False)

---
### Portfolio notes

- **Metrics to highlight:** test accuracy, macro F1 (class imbalance-aware), and the confusion
  matrix — call out which cell types are hardest to separate (e.g. visually similar WBC subtypes).
- **What to write up:** dataset composition (multi-source, 8 classes, ~46K images), why ViT
  (transfer learning from ImageNet-21k pretraining), preprocessing choices (augmentation for
  train, deterministic resize/crop for eval), and any class-imbalance handling.
- **Nice extensions:** log training curves with `report_to="tensorboard"`, add early stopping,
  try a smaller/faster backbone (e.g. `vit-tiny`/`deit`) as a speed baseline, or deploy the
  Gradio demo permanently on a free Hugging Face Space.
